# SOLSTICE quickstart

Load the released DIII-D surrogate models and predict: control
parameters -> plasma background -> EIRENE source terms. Weights
download automatically from GitHub Releases (a few MB).

Runs anywhere — no GPU or account needed.


In [ ]:
!pip -q install "solstice-fusion[models] @ git+https://github.com/ORNL-Fusion/solstice.git"


## State model: control parameters -> plasma background


In [ ]:
from solstice import hub

state = hub.load('pepc-diiid-state-v1')
params = {'ptot': 6e6,           # total input power pe + pi [W]
          'chi': 0.7,            # thermal diffusivity [m^2/s]
          'core_fueling': 3e20,
          'puff_D2': 1e21,       # D2 gas puff [atom/s]
          'dna': 0.5}            # particle diffusivity [m^2/s]
fields = state.predict(params)
print({k: f'{v.min():.3g} .. {v.max():.3g}' for k, v in fields.items()})


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

verts = np.stack([state.mesh.cell_corners_r.values,
                  state.mesh.cell_corners_z.values], axis=-1)
def plot(v, title, ax, log=False):
    a = np.log10(np.clip(v, 1e-30, None)) if log else v
    pc = PolyCollection(verts, array=a, cmap='viridis', edgecolor='none')
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal')
    ax.set_title(title); plt.colorbar(pc, ax=ax, shrink=0.8)

fig, axs = plt.subplots(1, 3, figsize=(13, 6))
plot(fields['te'], 'Te (eV)', axs[0])
plot(fields['ne'], 'log10 ne (m$^{-3}$)', axs[1], log=True)
plot(fields['prad'], 'log10 P$_{rad}$ (W/m$^3$)', axs[2], log=True)
plt.tight_layout()


## Sources model: plasma state -> EIRENE source terms


In [ ]:
sources = hub.load('pepc-diiid-sources-v1')
plasma = {k: fields[k] for k in sources.manifest['variables']['plasma_features']}
terms = sources.predict(plasma, params)

fig, axs = plt.subplots(1, 2, figsize=(9, 6))
for ax, name in zip(axs, ['sp', 'qe']):
    v = terms[name]; lim = np.abs(v).max()
    pc = PolyCollection(verts, array=v, cmap='RdBu_r', edgecolor='none', clim=(-lim, lim))
    ax.add_collection(pc); ax.autoscale(); ax.set_aspect('equal')
    ax.set_title(name); plt.colorbar(pc, ax=ax, shrink=0.8)
plt.tight_layout()


## Out-of-distribution requests warn

The bundles carry their training box; extrapolations are flagged.


In [ ]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    state.predict({**params, 'ptot': 40e6})   # far above the 2-16 MW training range
print(w[0].message)
